# Паттерн 8: Dynamic Spawning — порождение агентов на лету

Во всех предыдущих паттернах агенты определены заранее: у каждого фиксированный промпт, фиксированный набор инструментов, фиксированная роль. Dynamic Spawning ломает это ограничение. Здесь **координатор (Spawner) создаёт агентов на лету** — анализирует задачу и генерирует специализированных агентов с кастомными промптами под конкретные подзадачи. Отработали — уничтожились.

Ключевое отличие от похожих паттернов: в **Map-Reduce** один и тот же агент клонируется на разные данные (промпт одинаковый, входы разные). В **Dynamic Spawning** каждый агент уникален — его роль, промпт и фокус конструируются в рантайме исходя из анализа задачи. Количество агентов тоже определяется динамически — не структурой графа, а содержанием запроса.

```mermaid
graph LR
    START((START)) --> Spawner{{"🎯 Spawner / Planner<br/><i>порождает агентов</i>"}}
    Spawner -->|"Send(role=юрист)"| A1(["🔹 Agent α — юрист<br/><i>выполняет подзадачу</i>"])
    Spawner -->|"Send(role=финансист)"| A2(["🔹 Agent β — финансист<br/><i>выполняет подзадачу</i>"])
    Spawner -->|"Send(role=аналитик)"| A3(["🔹 Agent γ — аналитик<br/><i>выполняет подзадачу</i>"])
    A1 --> Aggregator(["📊 Aggregator<br/><i>собирает результаты</i>"])
    A2 --> Aggregator
    A3 --> Aggregator
    Aggregator --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class Spawner coordinator
    class A1,A2,A3 worker
    class Aggregator aggregator
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Два типа состояния

В Dynamic Spawning нужны **два отдельных состояния**. Основное (`SpawningState`) хранит задачу, спецификации порождённых агентов, собранные результаты и финальный ответ. Ключевое — поле `results` с редьюсером `operator.add`: каждый порождённый агент добавляет свой результат в общий список, и LangGraph автоматически объединяет их.

Второе состояние (`SpawnedAgentState`) — для самих динамических агентов. Оно содержит задачу, назначенную роль, кастомный промпт и фокус — всё то, что Spawner генерирует в рантайме. Это принципиальный момент: каждый агент получает **уникальную конфигурацию**, а не одинаковую.

In [3]:
llm = get_llm()

# --- Состояние для динамически порождённого агента ---
class SpawnedAgentState(TypedDict):
    task: str               # Общая задача
    role: str               # Роль агента (сгенерирована Spawner'ом)
    prompt: str             # Промпт агента (сгенерирован Spawner'ом)
    focus: str              # На чём сфокусироваться
    result: str             # Результат работы агента

# --- Основное состояние графа ---
class SpawningState(TypedDict):
    task: str
    agent_specs: list[dict]                        # Спецификации агентов от Spawner'а
    results: Annotated[list[str], operator.add]    # Результаты (собираются reducer'ом)
    final_answer: str

## Узел Spawner: анализ задачи и генерация спецификаций

Spawner — сердце паттерна. Он получает задачу, анализирует её и решает, каких специалистов нужно привлечь. Результат — JSON-массив спецификаций: для каждого агента указываются роль, фокус и системный промпт. Всё это генерируется LLM в рантайме.

Качество всей системы напрямую зависит от Spawner'а. Если он плохо декомпозирует задачу, порождённые агенты будут бесполезны. Поэтому для Spawner'а критически важен чёткий промпт и, в идеале, мощная модель. Fallback на два базовых агента — страховка на случай, если LLM вернёт невалидный JSON.

In [4]:
def spawner_node(state: SpawningState) -> dict:
    response = llm.invoke([
        SystemMessage(content="""Ты — координатор. Проанализируй задачу и определи,
каких специалистов нужно привлечь для её решения.

Верни JSON-массив (и ТОЛЬКО JSON, без markdown):
[
  {"role": "название роли", "focus": "на чём сфокусироваться", "prompt": "системный промпт для агента"},
  ...
]

Создай от 2 до 4 специалистов. Каждый должен покрывать отдельный аспект задачи."""),
        HumanMessage(content=state["task"])
    ])

    try:
        specs = json.loads(response.content.strip())
    except json.JSONDecodeError:
        # Fallback — два базовых агента
        specs = [
            {"role": "Аналитик", "focus": "ключевые факты", "prompt": "Ты — аналитик. Выдели ключевые факты."},
            {"role": "Критик", "focus": "риски и проблемы", "prompt": "Ты — критик. Найди слабые места и риски."},
        ]

    print(f"  [Spawner] Порождаю {len(specs)} агентов:")
    for spec in specs:
        print(f"    - {spec['role']}: {spec['focus']}")

    return {"agent_specs": specs}

## Fan-out через Send(): порождение агентов

Вот где происходит магия Dynamic Spawning. Функция `spawn_agents` — это **условное ребро**, которое возвращает список `Send`-объектов. Каждый `Send` создаёт отдельный экземпляр узла `spawned_agent` со своим уникальным состоянием: индивидуальной ролью, промптом и фокусом.

Это тот же механизм `Send()`, что используется в Map-Reduce, но с принципиальным отличием: там каждый `Send` содержит одинаковый промпт и разные данные, а здесь — **каждый `Send` несёт уникальное задание на роль**. LangGraph автоматически запускает все порождённые агенты параллельно и собирает результаты через редьюсер.

In [5]:
def spawn_agents(state: SpawningState) -> list[Send]:
    return [
        Send("spawned_agent", {
            "task": state["task"],
            "role": spec["role"],
            "prompt": spec["prompt"],
            "focus": spec["focus"],
            "result": "",
        })
        for spec in state["agent_specs"]
    ]

## Порождённый агент: выполнение с кастомным промптом

Узел `spawned_agent` — это единый шаблон, который получает **разное состояние** при каждом запуске. Промпт, роль и фокус приходят из `SpawnedAgentState`, сформированного Spawner'ом. Агент выполняет свою часть анализа и возвращает результат в формат основного состояния (`results`), чтобы редьюсер `operator.add` корректно собрал все ответы в один список.

Обратите внимание: узел определён один раз, но работает как N разных агентов — каждый экземпляр имеет уникальную «личность».

In [6]:
def spawned_agent_node(state: SpawnedAgentState) -> dict:
    response = llm.invoke([
        SystemMessage(content=state["prompt"]),
        HumanMessage(content=(
            f"Задача: {state['task']}\n"
            f"Твой фокус: {state['focus']}\n"
            f"Дай краткий, конкретный анализ (3-5 пунктов)."
        ))
    ])
    result = f"## {state['role']}\n{response.content}"
    print(f"  [{state['role']}] Анализ готов ({len(response.content)} символов)")
    return {"results": [result]}

## Агрегатор: синтез результатов

После того как все порождённые агенты завершили работу, их результаты автоматически собираются в `state["results"]` через редьюсер. Агрегатор получает все анализы и синтезирует единый ответ: выделяет главные выводы, отмечает совпадения и расхождения между мнениями специалистов.

Этот узел — аналог «редактора», который из черновиков нескольких экспертов собирает согласованный документ.

In [7]:
def aggregator_node(state: SpawningState) -> dict:
    all_results = "\n\n".join(state["results"])
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — редактор. Синтезируй общий ответ на основе анализов "
            "от нескольких специалистов. Выдели главные выводы, "
            "отметь где мнения совпадают и где расходятся."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\nАнализы специалистов:\n{all_results}")
    ])
    print(f"  [Агрегатор] Синтез готов ({len(response.content)} символов)")
    return {"final_answer": response.content}

## Сборка графа

Граф имеет три узла: `spawner`, `spawned_agent` и `aggregator`. Поток линейный: START -> spawner -> (fan-out через Send) -> spawned_agent (N экземпляров параллельно) -> aggregator -> END.

Условное ребро из `spawner` вызывает функцию `spawn_agents`, которая возвращает список `Send`. Каждый `Send` порождает отдельный экземпляр `spawned_agent`. Безусловное ребро из `spawned_agent` ведёт к `aggregator` — LangGraph дождётся завершения **всех** экземпляров, прежде чем передать управление агрегатору.

In [8]:
graph = StateGraph(SpawningState)

graph.add_node("spawner", spawner_node)
graph.add_node("spawned_agent", spawned_agent_node)
graph.add_node("aggregator", aggregator_node)

graph.add_edge(START, "spawner")
graph.add_conditional_edges("spawner", spawn_agents)
graph.add_edge("spawned_agent", "aggregator")
graph.add_edge("aggregator", END)

app = graph.compile()

## Запуск

Подаём задачу: оценить перспективы внедрения мультиагентных систем в банковском секторе. Spawner проанализирует тему и самостоятельно решит, каких специалистов породить — например, эксперта по банковским технологиям, специалиста по рискам, аналитика рынка. Для другой задачи (скажем, «проанализируй лицензионное соглашение») Spawner породил бы совершенно другой набор агентов.

In [9]:
print("=" * 60)
print("ПРИМЕР 8: Dynamic Spawning — агенты на лету")
print("=" * 60)

result = app.invoke({
    "task": "Оцени перспективы внедрения мультиагентных систем в банковском секторе",
    "agent_specs": [],
    "results": [],
    "final_answer": "",
})

print(f"\n  Порождено агентов: {len(result['agent_specs'])}")
print(f"  Собрано анализов: {len(result['results'])}")
print(f"  Финальный ответ: {result['final_answer'][:300]}...")

ПРИМЕР 8: Dynamic Spawning — агенты на лету


  [Spawner] Порождаю 2 агентов:
    - Аналитик: ключевые факты
    - Критик: риски и проблемы


  [Аналитик] Анализ готов (1177 символов)


  [Критик] Анализ готов (1516 символов)


  [Агрегатор] Синтез готов (4647 символов)

  Порождено агентов: 2
  Собрано анализов: 2
  Финальный ответ: Ниже — синтез оценки перспектив мультиагентных систем в банковском секторе на основе обоих анализов.

## Краткий вывод

**Перспективы есть, и они заметные, но внедрение будет ограниченным, поэтапным и в первую очередь вспомогательным.**  
Мультиагентные системы наиболее перспективны там, где есть:
-...


## Итоги

**Dynamic Spawning** — паттерн максимальной адаптивности. Система подстраивается под любую задачу, не ограничиваясь предопределённым набором ролей.

**Поток работы:** Task -> Spawner (анализ + генерация спецификаций) -> Send() (параллельное порождение N уникальных агентов) -> Aggregator (синтез) -> Result.

**Ключевые отличия от Map-Reduce:**
- Map-Reduce: один промпт, разные данные (клонирование)
- Dynamic Spawning: разные промпты, разные роли, разные фокусы (порождение уникальных агентов)

**Когда использовать:** задачи, где набор необходимых компетенций заранее неизвестен и определяется содержанием входных данных. Аналитические платформы, работающие с разными доменами. Системы, которые должны масштабироваться на новые типы задач без изменения кода.